In [ ]:
from collections import Counter

import numpy as np
import plotly.express as px
from IPython.display import clear_output

from playground_package.find_the_sub_game.code.bayes_game_state import BayesGameState
from playground_package.find_the_sub_game.code.map_utils import (
    generate_gaussian_blob_map,
    generate_random_map,  # noqa: F401
    generate_uniform_map,
    sample_map_for_sub_location,
)
from playground_package.find_the_sub_game.code.sensors import (
    PingSensor,
    PingSensorSpec,
)
from playground_package.find_the_sub_game.code.simulation import GameSimulator
from playground_package.find_the_sub_game.code.strategies import (
    EntropyBasedStrategy,
    MostLikelySquareStrategy,
    RandomSquareStrategy,
    SequentialSquareStrategy,  # noqa: F401
)
from playground_package.find_the_sub_game.code.types import ProbMap
from playground_package.find_the_sub_game.ui.figures import map_figure


In [ ]:
def plot_map(
        map_arr: ProbMap,
        replace: bool = False,
        highlight: tuple[int, int] | None = None,
        highlight_label: str | None = None) -> None:
    """Show a map in the notebook; `replace` clears the previous output first (for game loops)."""
    if replace:
        clear_output(wait=True)

    map_figure(
        map_arr,
        last_ping=highlight,
        last_ping_label=highlight_label,
        title="Where is the sub?",
    ).show()


# Game

In [ ]:
# Sanity checks for _calculate_distance
sensor_spec = PingSensorSpec()

assert sensor_spec._calculate_distance((2, 2), (2, 2)) == 0
assert sensor_spec._calculate_distance((2, 4), (2, 2)) == 2
assert sensor_spec._calculate_distance((0, 0), (2, 2)) == np.sqrt(8)

assert np.isclose(sensor_spec._calculate_distance((1, 1), (2, 2)), np.sqrt(2))
assert np.isclose(sensor_spec._calculate_distance((2, 1), (2, 2)), 1)
assert np.isclose(sensor_spec._calculate_distance((4, 0), (2, 2)), np.sqrt(8))

print("distance sanity checks passed")

In [ ]:
sensor_spec = PingSensorSpec()
map_arr = generate_uniform_map(5, 5)
sensor = PingSensor(sub_true_location=(2, 2), map_arr=map_arr, sensor_spec=sensor_spec)

print("pinging (2, 2) 100 times...")
print(Counter([sensor.ping((2, 2)) for i in range(100)]))
print("pinging (1, 1) 100 times...")
print(Counter([sensor.ping((1, 1)) for i in range(100)]))
print("pinging (0, 0) 100 times...")
print(Counter([sensor.ping((0, 0)) for i in range(100)]))

In [ ]:
# todo - move as a test case
# map = np.array([
#     [0.4, 0.1, 0.3, 0.2]
# ])
# test_entropy_strategy = EntropyBasedStrategy(sensor_spec=PingSensorSpec())
# p_table = test_entropy_strategy._prepare_calc_p_table(map.shape)
# test_entropy_strategy._calc_detect_and_non_detect_maps(map, p_table, (0, 0))


# (array([[9.26193922e-01, 7.23589001e-02, 8.68306802e-04, 5.78871201e-04]]),
#  array([[0.12223071, 0.11459129, 0.4579068 , 0.3052712 ]]),
#  np.float64(0.3455000000000001))

# Play the game

In [ ]:
# map_arr = generate_uniform_map(5, 5)
map_arr = generate_gaussian_blob_map(10, 15, n_blobs=3)
# map_arr = generate_random_map(10, 10)

sub_location = sample_map_for_sub_location(map_arr)

sensor_spec = PingSensorSpec()
sensor = PingSensor(sub_location, map_arr, sensor_spec=sensor_spec)
game_state = BayesGameState(sub_location, map_arr, sensor)

strategy = MostLikelySquareStrategy()
# strategy = EntropyBasedStrategy(sensor_spec=sensor_spec)
# strategy = SequentialSquareStrategy()
# strategy = RandomSquareStrategy()


plot_map(map_arr)

ping_coordinates = []
print(f"Sub position is {sub_location}")

count = 0
while True:
    user_input = input("Press Enter to advance the game state...")
    if user_input.lower() == 'q':
        print("Exiting the game.")
        break

    location_to_ping = strategy.choose_location(game_state.map_array)

    count += 1
    result = game_state.advance_state(location_to_ping)

    print(f"Pinged location: {result.pinged_location}, Outcome: {result.ping_outcome}, Probability: {result.pinged_location_prob:.4f}")
    ping_coordinates.append(result.pinged_location)

    # plot_map(game_state.map_array, replace=False, highlight=result.pinged_location, highlight_label=f"{result.ping_outcome.name} ({result.pinged_location_prob:.1%})")
    plot_map(game_state.map_array, replace=True, highlight=result.pinged_location, highlight_label=f"{result.ping_outcome.name} ({result.pinged_location_prob:.1%})")

    if result.game_won:
        print("Submarine found!")
        print(f"Number of pings: {count}")
        print(Counter(ping_coordinates).most_common())
        break

# Simulate and compare strategies

In [ ]:
SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # fixed order: strategy 1, 2, 3, 4


def display_results(results: dict[str, list[int | None]]) -> None:
      """Compare strategies: summary table, ECDF and box plot of pings to find the sub.

      `results` maps a strategy name to its per-game ping counts; None marks a game
      that hit the turn cap (lost).
      """
      print(f"{'strategy':<28}{'games':>6}{'lost':>6}{'mean ± SE':>16}{'std':>8}"
            f"{'P10':>6}{'P50':>6}{'P90':>6}{'P95':>6}{'max':>6}")
      rows = []
      for name, pings in results.items():
            won = np.array([p for p in pings if p is not None])
            lost = len(pings) - len(won)
            se = won.std() / np.sqrt(len(won))
            p10, p50, p90, p95 = np.percentile(won, [10, 50, 90, 95])
            mean_se = f"{won.mean():.1f} ± {se:.1f}"
            print(f"{name:<28}{len(pings):>6}{lost:>6}{mean_se:>16}{won.std():>8.1f}"
                  f"{p10:>6.0f}{p50:>6.0f}{p90:>6.0f}{p95:>6.0f}{won.max():>6}")
            rows += [{"strategy": name, "pings": int(p)} for p in won]

      colors = {"color_discrete_sequence": SERIES_COLORS, "template": "plotly_white"}

      fig = px.ecdf(rows, x="pings", color="strategy", **colors,
                  title="Share of games won within n pings (higher and further left is better)")
      fig.update_traces(line_width=2)
      fig.update_layout(yaxis_tickformat=".0%", yaxis_title="games won", xaxis_title="pings")
      fig.show()

      fig = px.histogram(rows, x="pings", color="strategy", facet_row="strategy", **colors,
                   title="Distribution of pings to find the sub")
      fig.update_layout(showlegend=False, bargap=0.05)
      fig.update_yaxes(matches=None)  # each strategy gets its own count scale
      fig.show()

      fig = px.box(rows, x="pings", y="strategy", color="strategy", points="outliers", **colors,
                  title="Pings to find the sub")
      fig.update_layout(showlegend=False, yaxis_title=None)
      fig.show()

# Compare different prior: correct one, uniform one and totally wrong one

In [ ]:
NUM_GAMES = 1000

def choose_location_based_map_probabilities(map_arr: ProbMap) -> tuple[int, int]:
    """Sub location drawn from the map: the prior is correct."""
    flat_index = np.random.choice(map_arr.size, p=map_arr.ravel())
    return np.unravel_index(flat_index, map_arr.shape)


def choose_location_uniformly(map_arr: ProbMap) -> tuple[int, int]:
    """Sub location drawn uniformly: the prior is wrong."""
    flat_index = np.random.choice(map_arr.size)
    return np.unravel_index(flat_index, map_arr.shape)


def choose_location_where_it_is_most_unlikely(map_arr: ProbMap) -> tuple[int, int]:
    """Sub location drawn from the flipped map: the prior is as wrong as possible."""
    flipped = map_arr.max() - map_arr  # peak square -> 0, emptiest squares -> highest
    flipped = flipped / flipped.sum()
    flat_index = np.random.choice(map_arr.size, p=flipped.ravel())
    return np.unravel_index(flat_index, map_arr.shape)


map_arr = generate_gaussian_blob_map(10, 15, n_blobs=3)


default_sensor = PingSensorSpec()
# noisy_sensor = PingSensorSpec(p_hit=0.5, p_near=0.3)  # weaker direct hit, stronger neighbor signal

experiments = {
    "Prior correct, default sensor": (
        [choose_location_based_map_probabilities(map_arr) for _ in range(NUM_GAMES)],
        default_sensor
    ),
    "No information/Uniform prior, default sensor": (
        [choose_location_uniformly(map_arr) for _ in range(NUM_GAMES)],
        default_sensor
    ),
    "Incorrect prior, default sensor": (
        [choose_location_where_it_is_most_unlikely(map_arr) for _ in range(NUM_GAMES)],
        default_sensor
    ),

    # "Prior correct, noisy sensor": (sample_from_map, noisy_sensor),
}

# Each factory gets the experiment's sensor spec from the simulator
strategies = {
    "Most Likely": lambda spec: MostLikelySquareStrategy(),
    "Random": lambda spec: RandomSquareStrategy(),
    "Entropy-Based": lambda spec: EntropyBasedStrategy(spec),
}


results = {}
for experiment_name, (trials, spec) in experiments.items():
    results[experiment_name] = {}

    simulator = GameSimulator(map=map_arr, trials=trials, sensor_spec=spec)

    for strategy_name, strategy_factory in strategies.items():
        strategy_results = simulator.run(strategy_factory)

        results[experiment_name][strategy_name] = strategy_results


# plot_map(map_arr)
for experiment_name, experiment_results in results.items():
    _, spec = experiments[experiment_name]
    print(f"\n=== {experiment_name}: {spec} ===")
    display_results(experiment_results)

# Streamlit mockup

In [ ]:
# map
map_arr = generate_gaussian_blob_map(10, 15, n_blobs=3)

# choose sampling method, it is either: choose_location_based_map_probabilities, choose_location_uniformly, choose_location_where_it_is_most_unlikely
sub_true_location = choose_location_based_map_probabilities(map_arr)

# sensor
sensor_spec = PingSensorSpec()
sensor = PingSensor(sub_true_location, map_arr, sensor_spec=sensor_spec)

most_likely_strategy = MostLikelySquareStrategy()
entropy_strategy = EntropyBasedStrategy(sensor_spec=PingSensorSpec())

game_state = BayesGameState(sub_location, map_arr, sensor)

plot_map(map_arr)

ping_coordinates = []
print(f"Sub position is {sub_location}") # Toggle show

count = 0
while True:
    most_likely_strategy_proposal = most_likely_strategy.choose_location(game_state.map_array)
    print(f"Choose: {most_likely_strategy_proposal} as it has the highest probability")

    current_entropy = entropy_strategy.calc_entropy(game_state.map_array)
    print(f"current entropy in the game is: {current_entropy}")

    entropy_strategy_proposal, information_gain = entropy_strategy.choose_location_detailed(game_state.map_array)
    print(f"Choose: {entropy_strategy_proposal} as gives most information ({information_gain} bits)")


    user_input = input("What is your choice, enter q to exit")
    if user_input.lower() == 'q':
        print("Exiting the game.")
        break

    # actual_choice = parse_user_input(user_input)
    actual_choice = most_likely_strategy_proposal

    count += 1
    result = game_state.advance_state(actual_choice)
    print(f"Pinged location: {result.pinged_location}, Outcome: {result.ping_outcome}, Probability: {result.pinged_location_prob:.4f}")
    ping_coordinates.append(result.pinged_location)

    # plot_map(game_state.map_array, replace=False, highlight=result.pinged_location, highlight_label=f"{result.ping_outcome.name} ({result.pinged_location_prob:.1%})")
    plot_map(game_state.map_array, replace=True, highlight=result.pinged_location, highlight_label=f"{result.ping_outcome.name} ({result.pinged_location_prob:.1%})")

    if result.game_won:
        print("Submarine found!")
        print(f"Number of pings: {count}")
        print(Counter(ping_coordinates).most_common())
        break